In [ ]:
%py
# PySpark script to mask the invoice_number in the d_product_revenue_clone table

from pyspark.sql.functions import col, when, length

# Load the d_product_revenue table with error handling
try:
    d_product_revenue_df = spark.read.table("purgo_databricks.qa_1.d_product_revenue")
except Exception as e:
    print(f"Error loading d_product_revenue table: {e}")
    raise

# Mask the last 4 digits of the invoice_number in the dataframe
masked_df = d_product_revenue_df.withColumn(
    "invoice_number",
    when(
        col("invoice_number").isNotNull() & (col("invoice_number").rlike("^[0-9A-Za-z]+$")) & (length(col("invoice_number")) >= 4),
        col("invoice_number").substr(1, length(col("invoice_number")) - 4) + "****"
    ).otherwise(col("invoice_number"))
)

# Drop the d_product_revenue_clone table if it exists
spark.sql("DROP TABLE IF EXISTS purgo_databricks.qa_1.d_product_revenue_clone")

# Write the masked data to the clone table in Delta format
masked_df.write.format("delta").saveAsTable("purgo_databricks.qa_1.d_product_revenue_clone")

# End of script